# Telecom Customer Data 
## Data Quality Assessment and Cleaning


### 1 · Setup & load

We keep an immutable copy of the raw data (df_raw) so the final before/after
comparison is grounded in the true original state. All cleaning happens on the
working copy `df`.

In [72]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

RAW_PATH = "test_datafile.csv"
df_raw = pd.read_csv(RAW_PATH)   # immutable reference
df = df_raw.copy()               # working copy we clean in place

print(f"Loaded {df_raw.shape[0]:,} rows  x  {df_raw.shape[1]} columns")
df_raw.head()

Loaded 5,050 rows  x  17 columns


,customer_id,age,gender,tenure_months,contract_type,monthly_charges,total_charges,internet_service,phone_service,avg_monthly_gb_used,num_support_tickets,avg_monthly_minutes,satisfaction_score,payment_method,num_additional_services,last_interaction_date,churned
0,TC-004711,32.87,Male,10.15,Month-to-month,69.24,656.42,DSL,Yes,11.70,4.00,324.00,7.80,bank transfer,2,2024-06-14,1
1,TC-000692,59.39,Female,3.45,Month-to-month,98.48,251.15,DSL,no,9.46,1.00,306.80,6.00,Electronic check,5,2024-06-23,1
2,TC-000066,62.34,male,1.39,Two year,94.35,120.78,Fiber optic,Yes,9.56,4.00,349.50,5.50,Bank transfer,0,2024-06-21,0
3,TC-003427,45.79,Female,67.61,Month-to-month,85.87,"5,834.73",Fiber optic,yes,3.15,1.00,258.20,4.70,Credit card,4,2024-06-21,1
4,TC-004821,39.63,F,27.32,One year,62.14,"1,626.23",DSL,Yes,28.80,0.00,335.80,12.30,Credit card,2,2024-06-19,0


In [73]:
# datatypes of each columns
df_raw.dtypes.to_frame("data_types")

,data_types
customer_id,object
age,float64
gender,object
tenure_months,float64
contract_type,object
monthly_charges,float64
total_charges,float64
internet_service,object
phone_service,object
avg_monthly_gb_used,float64


### 2 · Audit infrastructure

A small helpers power the whole notebook:

* `col_profile(frame)` — a per-column snapshot (dtype, # missing, # unique,
  numeric min/max). We snapshot the **raw** frame now; we snapshot the cleaned
  frame at the end. The two snapshots become the before/after table.


In [74]:
def col_profile(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in frame.columns:
        s = frame[c]
        rec = {
            "column": c,
            "dtype": str(s.dtype),
            "n_missing": int(s.isna().sum()),
            "n_unique": int(s.nunique(dropna=True)),
        }
        if pd.api.types.is_numeric_dtype(s) and s.notna().any():
            rec["min"] = round(float(s.min()), 2)
            rec["max"] = round(float(s.max()), 2)
        else:
            rec["min"] = rec["max"] = None
        rows.append(rec)
    return pd.DataFrame(rows).set_index("column")

profile_before = col_profile(df_raw)

audit = []  # ledger of every detection/decision
def log(column, issue, n, action):
    audit.append({"column": column, "issue": issue,
                  "n_affected": int(n), "action": action})

profile_before

,dtype,n_missing,n_unique,min,max
column,,,,,
customer_id,object,0,5000,NaN,NaN
age,float64,10,4706,-1.00,999.00
gender,object,50,8,NaN,NaN
tenure_months,float64,11,4757,-12.00,500.00
contract_type,object,0,3,NaN,NaN
monthly_charges,float64,12,3740,-50.00,"9,999.00"
total_charges,float64,23,4935,0.66,"218,681.58"
internet_service,object,505,5,NaN,NaN
phone_service,object,0,6,NaN,NaN


The raw profile already reveals the trouble spots at a glance:

* **Missing values** in 10 columns.
* `gender`, `internet_service`, `phone_service`, `payment_method` carry far
  more unique values than they semantically should (case/abbreviation noise).
* Numeric **min/max are physically impossible**: `age` min `-1` / max `999`,
  `tenure_months` `-12` / `500`, `monthly_charges` `-50` / `9999`,
  `satisfaction_score` `-1.4` / `99`, `total_charges` up to `218,681`.

### 3 · Exact duplicate rows

**Detection.** Count byte-identical rows across *all* columns.

**Decision.** Drop them, keeping the first occurrence. Fully identical records
(same `customer_id` and every field) carry no new information and would
double-weight those customers in any downstream model or aggregate.

In [75]:
n_dupes = int(df.duplicated().sum())
dup_ids = df.loc[df.duplicated(keep=False), "customer_id"].nunique()
print(f"Exact duplicate rows: {n_dupes}  (covering {dup_ids} customer_ids)")

df = df.drop_duplicates(keep="first").reset_index(drop=True)
log("<all columns>", "exact duplicate rows", n_dupes, "dropped (kept first)")
print(f"Rows after de-duplication: {len(df):,}")

Exact duplicate rows: 50  (covering 50 customer_ids)
Rows after de-duplication: 5,000


### 4 · Inconsistent categorical encodings

**Detection.** Inspect the value counts of every categorical column. The same
real-world category appears under several surface forms (different casing,
abbreviations).

**Decision.** Map each surface form to a single canonical label. We *do not*
touch genuine `NaN`s here and we preserve
genuinely distinct categories:



In [76]:
cat_cols = ["gender", "internet_service", "phone_service", "payment_method"]
for c in cat_cols:
    print(f"--- {c} (raw) ---")
    print(df[c].value_counts(dropna=False).to_string(), "\n")

--- gender (raw) ---
gender
Male      1734
Female    1710
F          408
M          390
female     212
MALE       200
male       199
Other       97
NaN         50 

--- internet_service (raw) ---
internet_service
Fiber optic    1703
DSL            1267
No              729
NaN             497
fiber           436
dsl             368 

--- phone_service (raw) ---
phone_service
Yes    2506
No     1225
yes     427
no      345
N       256
Y       241 

--- payment_method (raw) ---
payment_method
Credit card         1257
Electronic check    1203
Bank transfer       1018
Mailed check         734
credit card          241
bank transfer        192
BT                   164
CC                   161
NaN                   30 



In [77]:
def _norm(x):
    return x if pd.isna(x) else str(x).strip().lower()

GENDER_MAP   = {"m": "Male", "male": "Male", "f": "Female", "female": "Female",
                "other": "Other"}
INTERNET_MAP = {"fiber": "Fiber optic", "fiber optic": "Fiber optic",
                "fibre": "Fiber optic", "dsl": "DSL", "no": "No"}
PHONE_MAP    = {"y": "Yes", "yes": "Yes", "n": "No", "no": "No"}
PAYMENT_MAP  = {"cc": "Credit card", "credit card": "Credit card",
                "bt": "Bank transfer", "bank transfer": "Bank transfer",
                "electronic check": "Electronic check",
                "mailed check": "Mailed check"}

def standardize(col, mapping):
    """Map normalized values; count how many cells actually changed (NaN-safe)."""
    new = df[col].map(lambda x: np.nan if pd.isna(x) else mapping.get(_norm(x), x))
    sentinel = "\x00"  # placeholder so NaN==NaN compares as equal
    changed = int((new.fillna(sentinel) != df[col].fillna(sentinel)).sum())
    df[col] = new
    log(col, "inconsistent encoding", changed,
        f"standardized to {sorted(set(mapping.values()))}")
    return changed

for col, mp in [("gender", GENDER_MAP), ("internet_service", INTERNET_MAP),
                ("phone_service", PHONE_MAP), ("payment_method", PAYMENT_MAP)]:
    n = standardize(col, mp)
    print(f"{col:18s}: {n:4d} cells re-coded -> {sorted(df[col].dropna().unique())}")

gender            : 1409 cells re-coded -> ['Female', 'Male', 'Other']
internet_service  :  804 cells re-coded -> ['DSL', 'Fiber optic', 'No']
phone_service     : 1269 cells re-coded -> ['No', 'Yes']
payment_method    :  758 cells re-coded -> ['Bank transfer', 'Credit card', 'Electronic check', 'Mailed check']


### 5 · Impossible values & sentinel placeholders (numeric)

**Detection.** For each numeric column we define a *physically plausible domain*
and an explicit set of *sentinel* magic-numbers. Anything outside the domain is
**impossible**; the magic-numbers are **placeholders** that software writes when
a true value is unknown (classic `999`, `9999`, `500`).



**Decision.** Convert every impossible value and every sentinel to `NaN`. We
turn them into honest missing values rather than clipping to a boundary,
because a `-50` charge or a `999` age tells us the field is *unknown/corrupt*,
not that it equals the nearest legal value.

In [78]:
DOMAIN = {
    "age":                 dict(lo=18,  hi=100,    sentinels=[999]),
    "tenure_months":       dict(lo=0,   hi=120,    sentinels=[500]),
    "monthly_charges":     dict(lo=0,   hi=500,    sentinels=[9999], lo_strict=True),
    "total_charges":       dict(lo=0,   hi=100000, sentinels=[],     lo_strict=True),
    "avg_monthly_gb_used": dict(lo=0,   hi=None,   sentinels=[]),
    "num_support_tickets": dict(lo=0,   hi=100,    sentinels=[500]),
    "avg_monthly_minutes": dict(lo=0,   hi=None,   sentinels=[]),
    "satisfaction_score":  dict(lo=0,   hi=10,     sentinels=[99]),
}

for col, rule in DOMAIN.items():
    s = df[col]
    lo, hi = rule["lo"], rule["hi"]
    below = (s < lo) if not rule.get("lo_strict") else (s <= lo)
    impossible = below.fillna(False)
    if hi is not None:
        impossible = impossible | (s > hi).fillna(False)
    sent = s.isin(rule["sentinels"]) if rule["sentinels"] else pd.Series(False, index=s.index)

    # sentinel takes precedence in labeling; avoid double counting
    impossible = impossible & ~sent
    n_imp, n_sent = int(impossible.sum()), int(sent.sum())

    df.loc[impossible | sent, col] = np.nan
    if n_imp:
        log(col, "impossible (out of domain)", n_imp, f"-> NaN (domain [{lo},{hi}])")
    if n_sent:
        log(col, "sentinel placeholder", n_sent, f"-> NaN (values {rule['sentinels']})")
    print(f"{col:22s} impossible={n_imp:3d}  sentinel={n_sent:3d}")

age                    impossible= 19  sentinel=  1
tenure_months          impossible=  4  sentinel=  1
monthly_charges        impossible=  6  sentinel=  2
total_charges          impossible=  2  sentinel=  0
avg_monthly_gb_used    impossible= 10  sentinel=  0
num_support_tickets    impossible=  7  sentinel=  3
avg_monthly_minutes    impossible=  0  sentinel=  0
satisfaction_score     impossible=136  sentinel=  4


`Impossible` → unexpected bad data

`Sentinel`   → expected placeholder for missing data

 A subtle placeholder: the `age` = 18.00 spike

Beyond the obvious `999`, there is a **statistical** sentinel hiding in `age`.
The column is a *continuous float* (values like 32.87, 59.39), yet **261 records
sit at exactly 18.00** while neighbouring values (18.03, 18.08, …) occur only a
handful of times each. A real population does not pile up on a single integer
with a `.00` fraction in an otherwise-continuous field — this is a **default /
floor value** ("unknown adult → 18").

**Decision.** Treat `age == 18.00` as a placeholder → `NaN`.


In [79]:
# Evidence: fractional-part analysis around the spike
near18 = df_raw["age"].round(2).value_counts().reindex(
    [18.0, 18.03, 18.08, 18.44, 18.65]).fillna(0).astype(int)
print("age value counts near 18:\n", near18.to_string(), "\n")

mask18 = (df["age"] == 18.0).fillna(False)
n18 = int(mask18.sum())
df.loc[mask18, "age"] = np.nan
log("age", "sentinel placeholder", n18, "age==18.00 spike -> NaN (default/floor)")
print(f"Nulled {n18} placeholder ages (==18.00)")

# Contrast: monthly_charges==15 is left intact (documented decision)
n15 = int((df["monthly_charges"] == 15.0).sum())
print(f"monthly_charges==15.00 kept ({n15} rows) — legitimate plan price, not a sentinel")

age value counts near 18:
 age
18.00    261
18.03      2
18.08      3
18.44      3
18.65      2 

Nulled 260 placeholder ages (==18.00)
monthly_charges==15.00 kept (109 rows) — legitimate plan price, not a sentinel


### 6 · Semantic / cross-field outliers

These rows have values that are individually legal but **mutually
inconsistent**.



###  `total_charges` vs `monthly_charges` × `tenure_months`
For an active account, lifetime `total_charges` should be roughly
`monthly_charges × tenure_months`. We compute the ratio
`r = total_charges / (monthly_charges × tenure_months)` for rows where all three
inputs are valid and tenure ≥ 1.

**Decision.** Flag rows with `r < 0.3` or `r > 3` as semantically inconsistent
and null **only** `total_charges` for them (the field most prone to a stray
extra digit, e.g. the 218,681 value). We keep the row and its other fields. A
generous ±band avoids punishing normal variation from discounts, one-off fees,
or partial first/last months — the cluster sits tightly around 1.0 (IQR
≈ 0.94–1.07), so only genuine anomalies fall outside.

In [80]:
valid = (df["monthly_charges"].notna() & df["tenure_months"].notna()
         & df["total_charges"].notna() & (df["tenure_months"] >= 1)
         & (df["monthly_charges"] > 0))
expected = df["monthly_charges"] * df["tenure_months"]
ratio = (df["total_charges"] / expected).where(valid)

print("ratio total/(monthly*tenure) — distribution over valid rows:")
print(ratio.describe(percentiles=[.05, .25, .5, .75, .95]).to_string())

inconsistent = valid & ((ratio < 0.3) | (ratio > 3))
n_semantic = int(inconsistent.sum())
df.loc[inconsistent, "total_charges"] = np.nan
log("total_charges", "semantic outlier (vs monthly*tenure)", n_semantic,
    "-> NaN (ratio outside [0.3, 3])")
print(f"\nFlagged & nulled {n_semantic} semantically inconsistent total_charges")

ratio total/(monthly*tenure) — distribution over valid rows:
count   4,965.00
mean        1.05
std         0.48
min         0.01
5%          0.54
25%         0.94
50%         1.00
75%         1.07
95%         1.56
max         8.84

Flagged & nulled 184 semantically inconsistent total_charges


## 7 · Dates & residual missingness

**`last_interaction_date`** parses cleanly to dates spanning 2023-10 → 2025-06
(all in the past relative to today, no future timestamps), so we simply coerce
the dtype to `datetime64` and flag any unparseable strings.

**Missing-value strategy.** After part 3– part 6 we have a single, honest pool of `NaN`s
(original gaps + everything we invalidated). We produce two artefacts:

* **`df_clean`** — structurally clean, **missing values preserved as `NaN`.**
  This is the correct hand-off for most ML pipelines (which model missingness
  explicitly).
* **`df_final`** — a convenience copy with a *documented, reversible* imputation:
  * **Numeric** → column **median** (robust to the skew/outliers we just found).
  * **Categorical** (`gender`, `internet_service`, `payment_method`) → an
    explicit **`"Unknown"`** category. We refuse to invent a customer's gender
    or payment method by mode-filling; "Unknown" is the truthful label.

In [81]:
# Date coercion
parsed = pd.to_datetime(df["last_interaction_date"], errors="coerce")
n_bad_dates = int(parsed.isna().sum() - df["last_interaction_date"].isna().sum())
df["last_interaction_date"] = parsed

print(f"date range: {parsed.min().date()} -> {parsed.max().date()} "
      f"| unparseable: {n_bad_dates}")

df_clean = df.copy()  # NaNs preserved
residual_missing = df_clean.isna().sum()
print("\nResidual missing per column (df_clean):")
print(residual_missing[residual_missing > 0].to_string())

date range: 2023-10-23 -> 2025-06-09 | unparseable: 0

Residual missing per column (df_clean):
age                    290
gender                  50
tenure_months           15
monthly_charges         20
total_charges          208
internet_service       497
avg_monthly_gb_used     25
num_support_tickets     10
avg_monthly_minutes     80
satisfaction_score     160
payment_method          30


In [82]:
df_final = df_clean.copy()

num_cols = df_final.select_dtypes(include="number").columns
for c in num_cols:
    n_na = int(df_final[c].isna().sum())
    if n_na:
        df_final[c] = df_final[c].fillna(df_final[c].median())
        log(c, "missing value", n_na, "imputed with median")

for c in ["gender", "internet_service", "payment_method"]:
    n_na = int(df_final[c].isna().sum())
    if n_na:
        df_final[c] = df_final[c].fillna("Unknown")
        log(c, "missing value", n_na, "imputed with 'Unknown' category")

print("Remaining missing in df_final (numeric+key categoricals should be 0):")
print(df_final.isna().sum()[df_final.isna().sum() > 0].to_string() or "  none")

Remaining missing in df_final (numeric+key categoricals should be 0):
Series([], )


## 8 · Before / after summary

### 8.1 · The audit ledger
Every detection/decision the pipeline made, as data:

In [83]:
audit_df = pd.DataFrame(audit)
audit_df

,column,issue,n_affected,action
0,<all columns>,exact duplicate rows,50,dropped (kept first)
1,gender,inconsistent encoding,1409,"standardized to ['Female', 'Male', 'Other']"
2,internet_service,inconsistent encoding,804,"standardized to ['DSL', 'Fiber optic', 'No']"
3,phone_service,inconsistent encoding,1269,"standardized to ['No', 'Yes']"
4,payment_method,inconsistent encoding,758,"standardized to ['Bank transfer', 'Credit card..."
5,age,impossible (out of domain),19,"-> NaN (domain [18,100])"
6,age,sentinel placeholder,1,-> NaN (values [999])
7,tenure_months,impossible (out of domain),4,"-> NaN (domain [0,120])"
8,tenure_months,sentinel placeholder,1,-> NaN (values [500])
9,monthly_charges,impossible (out of domain),6,"-> NaN (domain [0,500])"


In [84]:
# Cells affected, rolled up per column
issues_by_col = (audit_df.groupby("column")["n_affected"].sum()
                 .sort_values(ascending=False).rename("cells_flagged"))
issues_by_col.to_frame()

,cells_flagged
column,
gender,1459
internet_service,1301
phone_service,1269
payment_method,788
age,570
total_charges,394
satisfaction_score,300
avg_monthly_minutes,80
<all columns>,50


### 8.2 · Per-column before/after profile
Raw (`df_raw`) vs structurally-clean (`df_clean`). Note `n_missing_after`
*rises* for columns where we converted impossible/sentinel/semantic values into
honest `NaN`s — that increase is the audit working as intended, not data loss.

In [85]:
profile_after = col_profile(df_clean)
summary = profile_before.join(profile_after, lsuffix="_before", rsuffix="_after")
summary = summary[[
    "dtype_before", "dtype_after",
    "n_unique_before", "n_unique_after",
    "n_missing_before", "n_missing_after",
    "min_before", "min_after", "max_before", "max_after",
]]
summary["cells_flagged"] = issues_by_col.reindex(summary.index).fillna(0).astype(int)
summary

,dtype_before,dtype_after,n_unique_before,n_unique_after,n_missing_before,n_missing_after,min_before,min_after,max_before,max_after,cells_flagged
column,,,,,,,,,,,
customer_id,object,object,5000,5000,0,0,NaN,NaN,NaN,NaN,0
age,float64,float64,4706,4698,10,290,-1.00,18.03,999.00,85.00,570
gender,object,object,8,3,50,50,NaN,NaN,NaN,NaN,1459
tenure_months,float64,float64,4757,4754,11,15,-12.00,1.00,500.00,120.00,20
contract_type,object,object,3,3,0,0,NaN,NaN,NaN,NaN,0
monthly_charges,float64,float64,3740,3735,12,20,-50.00,15.00,"9,999.00",150.00,28
total_charges,float64,float64,4935,4755,23,208,0.66,6.26,"218,681.58","36,026.72",394
internet_service,object,object,5,3,505,497,NaN,NaN,NaN,NaN,1301
phone_service,object,object,6,2,0,0,NaN,NaN,NaN,NaN,1269


In [86]:
print("="*60)
print("DATA QUALITY AUDIT — SUMMARY")
print("="*60)
print(f"Raw rows                : {len(df_raw):,}")
print(f"Duplicate rows removed  : {n_dupes}")
print(f"Clean rows              : {len(df_clean):,}")
print(f"Total cells flagged     : {int(audit_df['n_affected'].sum()):,}")
print("-"*60)
print("Cells flagged by issue class:")
print(audit_df.groupby("issue")["n_affected"].sum()
      .sort_values(ascending=False).to_string())
print("="*60)

DATA QUALITY AUDIT — SUMMARY
Raw rows                : 5,050
Duplicate rows removed  : 50
Clean rows              : 5,000
Total cells flagged     : 6,314
------------------------------------------------------------
Cells flagged by issue class:
issue
inconsistent encoding                   4240
missing value                           1385
sentinel placeholder                     271
impossible (out of domain)               184
semantic outlier (vs monthly*tenure)     184
exact duplicate rows                      50


### 9 · Persist cleaned outputs

* `cleaned_data.csv` — `df_clean` (NaNs preserved; recommended for modelling).
* `cleaned_data_imputed.csv` — `df_final` (median / `"Unknown"` imputation).
* `data_quality_audit_log.csv` — the full ledger for traceability.

In [87]:
df_clean.to_csv("cleaned_data.csv", index=False)
df_final.to_csv("cleaned_data_imputed.csv", index=False)
audit_df.to_csv("data_quality_audit_log.csv", index=False)
print("Wrote: cleaned_data.csv, cleaned_data_imputed.csv, data_quality_audit_log.csv")

Wrote: cleaned_data.csv, cleaned_data_imputed.csv, data_quality_audit_log.csv
